# Geometric SVM Threshold Sweep

This notebook is a descriptive, notebook-friendly version of the threshold-tuned LinearSVC experiment. [File src\training\geom-svm-thresholdTuneClassifier-csweep.py]

The workflow is organized into phases:
- environment and imports
- data loading and preprocessing configuration
- feature generation
- train/holdout splitting and cross-validation
- class balancing and threshold tuning
- final holdout evaluation, plots, W&B logging, and model export

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'train.csv').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the current working directory.')

project_root = find_project_root(Path.cwd())
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocessing.one_step import OneStepOptions, preprocess_one_step
from submit.save_model import save_model

print(f'Project root: {project_root}')

## Data Loading and Experiment Settings

This cell loads the training data and defines the preprocessing switches that control how the one-step feature pipeline behaves.

The `df` guard matters in notebooks because cells can be re-run out of order. If `df` already exists but is no longer a DataFrame, the notebook reloads the source CSV.

In [ ]:
wandb.login()
df = None

if 'df' not in globals() or not isinstance(df, pd.DataFrame):
    df = pd.read_csv(project_root / 'data' / 'train.csv')

print(type(df))
print(df.shape)
df.head()

noemp_option = 'C'
newexist_option = 'A'
createjob_option = 'C'
retainedjob_option = 'B'
disbursementgross_option = 'C'
approvaldate_option = 'A'
approvalfy_option = 'A'
franchise_option = 'binary'
urbanrural_option = 'onehot'
revlinecr_option = 'C'
lowdoc_option = 'C'
local_state = 'IL'

options = OneStepOptions(
    noemp_option=noemp_option,
    newexist_option=newexist_option,
    createjob_option=createjob_option,
    retainedjob_option=retainedjob_option,
    approvaldate_option=approvaldate_option,
    approvalfy_option=approvalfy_option,
    franchise_option=franchise_option,
    urbanrural_option=urbanrural_option,
    revlinecr_option=revlinecr_option,
    lowdoc_option=lowdoc_option,
    disbursementgross_option=disbursementgross_option,
    local_state=local_state,
)

## Feature Construction

The one-step preprocessing function turns the raw loan data into a modeling table. This is the right place to inspect feature count, row count, and whether the target column is present before building splits.

In [ ]:
df_processed = preprocess_one_step(df, options=options)
print(f'Rows: {len(df_processed):,}')
print(f'Features: {df_processed.shape[1]}')
df_processed.head()

## Train / Holdout Split and Cross-Validation Plan

The holdout set is reserved for final reporting only. Cross-validation is run on the train/validation portion so that threshold tuning and C sweeps do not leak information from the holdout split.

In [ ]:
target_col = 'Accept'
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

print(f'Dataset shape: {X.shape}')
print('Target distribution:')
print(y.value_counts())
print()

X_trainval, X_holdout, y_trainval, y_holdout = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Train/Val set size: {X_trainval.shape[0]}')
print(f'Holdout set size: {X_holdout.shape[0]}')
print('Train/Val target distribution:')
print(y_trainval.value_counts())
print()
print('Holdout target distribution:')
print(y_holdout.value_counts())
print()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'StratifiedKFold splits: {skf.get_n_splits()}')
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval), 1):
    print(f'Fold {fold_idx}: Train={len(train_idx)}, Val={len(val_idx)}')
n_splits = skf.get_n_splits()

## Class Balance and Threshold Utilities

The classifier can be trained with class weights, over-sampling, or under-sampling. Threshold tuning is done on out-of-fold decision scores so the final cutoff is chosen from cross-validated predictions rather than from the holdout set.

In [ ]:
balance_strategy = 'class_weight'
reject_class_weight = 2.0
optimize_metric = 'macro_f1'
enable_threshold_tuning = True
use_scaler = False
candidate_c_values = [0.01, 0.1, 0.3, 1.0, 3.0, 10.0]

def rebalance_training_data(X_train, y_train, strategy: str, random_state: int = 42):
    if strategy == 'none':
        return X_train, y_train

    train_frame = X_train.copy()
    train_frame['Accept'] = y_train.values

    reject_mask = train_frame['Accept'] == 0
    reject_rows = train_frame[reject_mask]
    approve_rows = train_frame[~reject_mask]

    if reject_rows.empty or approve_rows.empty:
        return X_train, y_train

    if strategy == 'oversample_reject':
        reject_rows = reject_rows.sample(n=len(approve_rows), replace=True, random_state=random_state)
        balanced = pd.concat([approve_rows, reject_rows], ignore_index=True)
    elif strategy == 'undersample_approve':
        if len(approve_rows) <= len(reject_rows):
            return X_train, y_train
        approve_rows = approve_rows.sample(n=len(reject_rows), replace=False, random_state=random_state)
        balanced = pd.concat([approve_rows, reject_rows], ignore_index=True)
    else:
        raise ValueError(
            f'Unknown balance_strategy: {strategy}. Use none, class_weight, oversample_reject, or undersample_approve.'
        )

    balanced = balanced.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return balanced.drop(columns=['Accept']), balanced['Accept']

def predict_with_threshold(scores: np.ndarray, threshold: float) -> np.ndarray:
    return (scores >= threshold).astype(int)

def score_threshold(y_true: pd.Series, y_pred: np.ndarray, metric_name: str) -> float:
    if metric_name == 'balanced_accuracy':
        return balanced_accuracy_score(y_true, y_pred)
    if metric_name == 'macro_f1':
        return f1_score(y_true, y_pred, average='macro', zero_division=0)
    if metric_name == 'mcc':
        return matthews_corrcoef(y_true, y_pred)
    raise ValueError(f'Unknown optimize_metric: {metric_name}')

## Model Configuration and W&B Run

This cell declares the sweep settings and opens a single W&B run for the experiment. The configuration is intentionally explicit so the resulting run is easy to compare with earlier geometric and bagging experiments.

In [ ]:
class_weight = {0: reject_class_weight, 1: 1.0} if balance_strategy == 'class_weight' else None

run = wandb.init(
    project='MS Geometric - SVM',
    config={
        'model_name': 'LinearSVC_threshold_tuning_csweep',
        'random_state': 42,
        'max_iter': 10000,
        'use_scaler': use_scaler,
        'balance_strategy': balance_strategy,
        'reject_class_weight': reject_class_weight,
        'optimize_metric': optimize_metric,
        'enable_threshold_tuning': enable_threshold_tuning,
        'candidate_c_values': candidate_c_values,
        'noemp_option': noemp_option,
        'newexist_option': newexist_option,
        'createjob_option': createjob_option,
        'retainedjob_option': retainedjob_option,
        'approvaldate_option': approvaldate_option,
        'approvalfy_option': approvalfy_option,
        'franchise_option': franchise_option,
        'urbanrural_option': urbanrural_option,
        'revlinecr_option': revlinecr_option,
        'lowdoc_option': lowdoc_option,
        'disbursementgross_option': disbursementgross_option,
        'local_state': local_state,
        'cv_n_splits': n_splits,
        'n_train_rows': int(X_trainval.shape[0]),
        'n_holdout_rows': int(X_holdout.shape[0]),
        'n_features': int(X_trainval.shape[1]),
    },
)

## LinearSVC Sweep and Threshold Tuning

For each candidate `C`, the notebook runs stratified cross-validation, collects fold metrics, and optionally tunes the decision threshold using the out-of-fold decision scores. The selected operating point is the one with the best out-of-fold objective, with ROC-AUC used as a tie-breaker.

In [ ]:
def build_pipeline(c_value: float) -> Pipeline:
    return Pipeline(
        steps=[
            ('scaler', StandardScaler() if use_scaler else 'passthrough'),
            (
                'model',
                LinearSVC(
                    random_state=42,
                    max_iter=10000,
                    C=c_value,
                    class_weight=class_weight,
                ),
            ),
        ]
    )

cv_results_by_c = []

for c_value in candidate_c_values:
    cv_fold_metrics = []
    oof_true = []
    oof_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval), 1):
        X_fold_train = X_trainval.iloc[train_idx]
        y_fold_train = y_trainval.iloc[train_idx]
        X_fold_val = X_trainval.iloc[val_idx]
        y_fold_val = y_trainval.iloc[val_idx]

        if balance_strategy in {'oversample_reject', 'undersample_approve'}:
            X_fold_train, y_fold_train = rebalance_training_data(
                X_fold_train,
                y_fold_train,
                strategy=balance_strategy,
                random_state=42,
            )

        fold_pipeline = build_pipeline(c_value)
        fold_pipeline.fit(X_fold_train, y_fold_train)

        y_fold_pred = fold_pipeline.predict(X_fold_val)
        y_fold_score = fold_pipeline.decision_function(X_fold_val)
        oof_true.append(y_fold_val)
        oof_scores.append(y_fold_score)

        fold_metrics = {
            'fold': fold_idx,
            'roc_auc': roc_auc_score(y_fold_val, y_fold_score),
            'pr_auc': average_precision_score(y_fold_val, y_fold_score),
            'f1': f1_score(y_fold_val, y_fold_pred, zero_division=0),
            'precision': precision_score(y_fold_val, y_fold_pred, zero_division=0),
            'recall': recall_score(y_fold_val, y_fold_pred, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_fold_val, y_fold_pred),
            'macro_f1': f1_score(y_fold_val, y_fold_pred, average='macro', zero_division=0),
            'mcc': matthews_corrcoef(y_fold_val, y_fold_pred),
            'accuracy': accuracy_score(y_fold_val, y_fold_pred),
        }
        cv_fold_metrics.append(fold_metrics)

    cv_summary = {
        'C': float(c_value),
        'cv_mean_roc_auc': float(np.mean([m['roc_auc'] for m in cv_fold_metrics])),
        'cv_std_roc_auc': float(np.std([m['roc_auc'] for m in cv_fold_metrics])),
        'cv_mean_pr_auc': float(np.mean([m['pr_auc'] for m in cv_fold_metrics])),
        'cv_std_pr_auc': float(np.std([m['pr_auc'] for m in cv_fold_metrics])),
        'cv_mean_f1': float(np.mean([m['f1'] for m in cv_fold_metrics])),
        'cv_std_f1': float(np.std([m['f1'] for m in cv_fold_metrics])),
        'cv_mean_precision': float(np.mean([m['precision'] for m in cv_fold_metrics])),
        'cv_std_precision': float(np.std([m['precision'] for m in cv_fold_metrics])),
        'cv_mean_recall': float(np.mean([m['recall'] for m in cv_fold_metrics])),
        'cv_std_recall': float(np.std([m['recall'] for m in cv_fold_metrics])),
        'cv_mean_balanced_accuracy': float(np.mean([m['balanced_accuracy'] for m in cv_fold_metrics])),
        'cv_std_balanced_accuracy': float(np.std([m['balanced_accuracy'] for m in cv_fold_metrics])),
        'cv_mean_macro_f1': float(np.mean([m['macro_f1'] for m in cv_fold_metrics])),
        'cv_std_macro_f1': float(np.std([m['macro_f1'] for m in cv_fold_metrics])),
        'cv_mean_mcc': float(np.mean([m['mcc'] for m in cv_fold_metrics])),
        'cv_std_mcc': float(np.std([m['mcc'] for m in cv_fold_metrics])),
        'cv_mean_accuracy': float(np.mean([m['accuracy'] for m in cv_fold_metrics])),
        'cv_std_accuracy': float(np.std([m['accuracy'] for m in cv_fold_metrics])),
    }

    decision_threshold = 0.0
    tuned_objective = cv_summary[f'cv_mean_{optimize_metric}']
    if enable_threshold_tuning:
        y_oof_true = pd.concat(oof_true).reset_index(drop=True)
        y_oof_scores = np.concatenate(oof_scores)
        threshold_grid = np.quantile(y_oof_scores, np.linspace(0.05, 0.95, 121))

        best_threshold = 0.0
        best_threshold_score = -np.inf
        for threshold in threshold_grid:
            y_oof_pred = predict_with_threshold(y_oof_scores, float(threshold))
            metric_value = score_threshold(y_oof_true, y_oof_pred, optimize_metric)
            if metric_value > best_threshold_score:
                best_threshold_score = metric_value
                best_threshold = float(threshold)

        decision_threshold = best_threshold
        tuned_objective = float(best_threshold_score)
        cv_summary['cv_oof_threshold'] = decision_threshold
        cv_summary['cv_oof_objective'] = tuned_objective

    cv_results_by_c.append((c_value, decision_threshold, tuned_objective, cv_summary, cv_fold_metrics))

best_c_value, decision_threshold, best_oof_objective, cv_summary, cv_fold_metrics = max(
    cv_results_by_c,
    key=lambda item: (item[2], item[3]['cv_mean_roc_auc']),
)

print('Cross-validation summary')
for c_value, threshold_value, objective_value, summary, _ in cv_results_by_c:
    print(
        f'C={c_value:g} | ' +
        f'CV MACRO-F1={summary["cv_mean_macro_f1"]:.4f} +/- {summary["cv_std_macro_f1"]:.4f} ' +
        f'CV ACC={summary["cv_mean_accuracy"]:.4f} +/- {summary["cv_std_accuracy"]:.4f} ' +
        f'Threshold={threshold_value:.5f} ' +
        f'OOF {optimize_metric}={objective_value:.4f}'
    )

print(
    f'Best C selected by OOF {optimize_metric}: {best_c_value:g} ' +
    f'(threshold={decision_threshold:.5f}, score={best_oof_objective:.4f})'
)

for metric_name in [
    'roc_auc',
    'pr_auc',
    'f1',
    'precision',
    'recall',
    'balanced_accuracy',
    'macro_f1',
    'mcc',
    'accuracy',
]:
    print(
        f'CV {metric_name.upper()}: ' +
        f'{cv_summary[f"cv_mean_{metric_name}"]:.4f} +/- ' +
        f'{cv_summary[f"cv_std_{metric_name}"]:.4f}'
    )

## Holdout Evaluation

After the best `C` and threshold are selected, the model is refit on the full train/validation split and evaluated once on the untouched holdout data. This is the main estimate to compare with other experiments.

In [ ]:
cv_table = wandb.Table(
    columns=['fold', 'roc_auc', 'pr_auc', 'f1', 'macro_f1', 'precision', 'recall', 'accuracy'],
    data=[
        [
            int(m['fold']),
            float(m['roc_auc']),
            float(m['pr_auc']),
            float(m['f1']),
            float(m['macro_f1']),
            float(m['precision']),
            float(m['recall']),
            float(m['accuracy']),
        ]
        for m in cv_fold_metrics
    ],
)

csweep_table = wandb.Table(
    columns=[
        'C',
        'threshold',
        'oof_objective',
        'cv_mean_macro_f1',
        'cv_mean_accuracy',
        'cv_mean_roc_auc',
        'cv_mean_pr_auc',
    ],
    data=[
        [
            float(c_value),
            float(threshold_value),
            float(objective_value),
            float(summary['cv_mean_macro_f1']),
            float(summary['cv_mean_accuracy']),
            float(summary['cv_mean_roc_auc']),
            float(summary['cv_mean_pr_auc']),
        ]
        for c_value, threshold_value, objective_value, summary, _ in cv_results_by_c
    ],
)

wandb.log(
    {
        'cv/folds_table': cv_table,
        'cv/csweep_table': csweep_table,
        'selected_c': best_c_value,
        'threshold/selected': decision_threshold,
        'threshold/oof_objective': best_oof_objective,
        **cv_summary,
    }
)

if balance_strategy in {'oversample_reject', 'undersample_approve'}:
    X_trainval_fit, y_trainval_fit = rebalance_training_data(
        X_trainval,
        y_trainval,
        strategy=balance_strategy,
        random_state=42,
    )
else:
    X_trainval_fit, y_trainval_fit = X_trainval, y_trainval

svm_pipeline = build_pipeline(best_c_value)
svm_pipeline.fit(X_trainval_fit, y_trainval_fit)

y_score = svm_pipeline.decision_function(X_holdout)
y_pred = predict_with_threshold(y_score, decision_threshold)

metrics = {
    'ROC-AUC': roc_auc_score(y_holdout, y_score),
    'PR-AUC': average_precision_score(y_holdout, y_score),
    'F1': f1_score(y_holdout, y_pred, zero_division=0),
    'Precision': precision_score(y_holdout, y_pred, zero_division=0),
    'Recall': recall_score(y_holdout, y_pred, zero_division=0),
    'Balanced-Accuracy': balanced_accuracy_score(y_holdout, y_pred),
    'Macro-F1': f1_score(y_holdout, y_pred, average='macro', zero_division=0),
    'MCC': matthews_corrcoef(y_holdout, y_pred),
    'Accuracy': accuracy_score(y_holdout, y_pred),
}

cm = confusion_matrix(y_holdout, y_pred)
report = classification_report(y_holdout, y_pred, output_dict=True)
positive_rate = float((y_pred == 1).mean())
score_mean = float(y_score.mean())
score_std = float(y_score.std())

print(f'Use StandardScaler: {use_scaler}')
print(f'Decision threshold: {decision_threshold:.5f}')
for name, value in metrics.items():
    print(f'{name}: {value:.4f}')
print(f'Predicted positive rate: {positive_rate:.4f}')

## Diagnostic Curves and Experiment Logging

The ROC and precision-recall plots help compare ranking quality and operating-point behavior. The final cell also logs the summary metrics, confusion matrix, classification report, and model artifact to W&B before saving the fitted pipeline locally for submission use.

In [ ]:
fpr, tpr, _ = roc_curve(y_holdout, y_score)
precision, recall, _ = precision_recall_curve(y_holdout, y_score)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {metrics['ROC-AUC']:.4f}")
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.7)
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

axes[1].plot(recall, precision, label=f"PR-AUC = {metrics['PR-AUC']:.4f}")
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.show()

wandb.log(
    {
        'roc_auc': metrics['ROC-AUC'],
        'pr_auc': metrics['PR-AUC'],
        'f1': metrics['F1'],
        'macro_f1': metrics['Macro-F1'],
        'precision': metrics['Precision'],
        'recall': metrics['Recall'],
        'balanced_accuracy': metrics['Balanced-Accuracy'],
        'mcc': metrics['MCC'],
        'accuracy': metrics['Accuracy'],
        'decision_threshold': decision_threshold,
        'predicted_positive_rate': positive_rate,
        'decision_score_mean': score_mean,
        'decision_score_std': score_std,
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }
)

run.summary['macro_f1'] = metrics['Macro-F1']
run.summary['holdout/macro_f1'] = metrics['Macro-F1']

wandb.log(
    {
        'confusion_matrix': wandb.plot.confusion_matrix(
            y_true=y_holdout.tolist(),
            preds=y_pred.tolist(),
            class_names=['Reject', 'Accept'],
        ),
        'roc_pr_curves': wandb.Image(fig),
    }
)

report_rows = []
for label, values in report.items():
    if isinstance(values, dict):
        report_rows.append(
            [
                label,
                float(values.get('precision', 0.0)),
                float(values.get('recall', 0.0)),
                float(values.get('f1-score', 0.0)),
                float(values.get('support', 0.0)),
            ]
        )

report_table = wandb.Table(
    columns=['label', 'precision', 'recall', 'f1_score', 'support'],
    data=report_rows,
)
wandb.log({'classification_report': report_table})

run.finish()

saved_paths = save_model(
    model_pipeline=svm_pipeline,
    preprocessing_options=options,
    feature_names=X_trainval.columns.tolist(),
    project_root=project_root,
    model_name='geom-svm-thresholdTuneClassifier-csweep',
)

saved_paths

## Notes on Expected Results

The best observed run in the source script favored a very small `C` and a low positive threshold. If the notebook reproduces that trend, the most likely interpretation is that a more conservative approval cutoff is improving macro-F1 without a large hit to ranking quality.

If you want to continue exploring, the most useful next step is usually a narrower `C` sweep around the best value and a finer threshold grid around the selected operating point.